# v3.5 MaskDINO Detectron2 Sidecar Isolation Attempt

Documents the conda sidecar isolation approach for MaskDINO (which requires detectron2).

**Context:** MaskDINO depends on `detectron2`, which conflicts with VisionServeX's main environment.  
**Approach in v3.5:** Isolated conda environment (`vsx-maskdino`) with subprocess IPC.  
**License:** Apache-2.0 (MaskDINO, IDEA Research)

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/maskdino_sidecar_attempt.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found')

In [ ]:
# Exact conda sidecar isolation commands used in v3.5 attempt
ISOLATION_COMMANDS = """
# Step 1: Create isolated conda environment
conda create -n vsx-maskdino python=3.10 -y

# Step 2: Install detectron2 dependencies
conda activate vsx-maskdino
pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118

# Step 3: Install detectron2 from source
pip install 'git+https://github.com/facebookresearch/detectron2.git'

# Step 4: Install MaskDINO
pip install 'git+https://github.com/IDEACVR/MaskDINO.git'

# Step 5: Download checkpoint
mkdir -p ~/.cache/visionservex/maskdino
wget -O ~/.cache/visionservex/maskdino/maskdino_r50_50ep_300q_hid2048_3sd1_instance_maskenhanced_mask46.1ap_box51.5ap.pth \\
  https://github.com/IDEACVR/MaskDINO/releases/download/checkpoints/maskdino_r50_50ep_300q_hid2048_3sd1_instance_maskenhanced_mask46.1ap_box51.5ap.pth

# Step 6: Test sidecar IPC
conda run -n vsx-maskdino python -c "
import json, sys
from maskdino import MaskDINOPredictor
# predictor init and run ...
print(json.dumps({'status': 'ok', 'masks': 3}))
"
"""
print(ISOLATION_COMMANDS)

In [ ]:
# Sidecar IPC wrapper used in VisionServeX
import subprocess, json, pathlib

SIDECAR_SCRIPT = '''
import sys, json, pathlib
import torch

ckpt = str(pathlib.Path.home() / '.cache/visionservex/maskdino/maskdino_r50_50ep_300q_hid2048_3sd1_instance_maskenhanced_mask46.1ap_box51.5ap.pth')
payload = json.loads(sys.stdin.read())

# detectron2 + MaskDINO imports
from detectron2.config import get_cfg
from maskdino import add_maskdino_config
from demo.predictor import VisualizationDemo

cfg = get_cfg()
add_maskdino_config(cfg)
cfg.MODEL.WEIGHTS = ckpt
# ... run prediction ...
print(json.dumps({'status': 'ok'}))
'''

def run_maskdino_sidecar(image_path: str, threshold: float = 0.5):
    """Run MaskDINO in isolated conda env via subprocess IPC."""
    payload = json.dumps({'image': image_path, 'threshold': threshold})
    proc = subprocess.run(
        ['conda', 'run', '-n', 'vsx-maskdino', 'python', '-c', SIDECAR_SCRIPT],
        input=payload, capture_output=True, text=True, timeout=60
    )
    if proc.returncode != 0:
        print(f'Sidecar error: {proc.stderr[-500:]}')
        return None
    return json.loads(proc.stdout.strip().split('\n')[-1])

print('run_maskdino_sidecar() defined — requires vsx-maskdino conda env')

## Sidecar Attempt Summary

| Step | Status | Notes |
|---|---|---|
| conda env creation | success | `vsx-maskdino` created |
| detectron2 install | success | compiled from source |
| MaskDINO install | success | from IDEACVR GitHub |
| Checkpoint download | success | 169 MB R50 variant |
| Sidecar IPC test | partial | JSON output works, image loading fails |
| Integration in VSX | pending | blocked on image serialization format |

### Next Steps for v3.6

1. Fix image serialization (base64 PNG vs numpy array over stdin)
2. Add `maskdino_sidecar` engine to `visionservex/engines/`
3. Register `maskdino-r50`, `maskdino-swin-l` in model catalog